<a href="https://colab.research.google.com/github/pikey-msc/RiesgosFinancieros/blob/master/2026-1/Swaps_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔄 Valuación de Swaps de Tasa de Interés (IRS)

**Riesgos Financieros — 2026-1**

Un **swap de tasas de interés** (IRS, *Interest Rate Swap*) es un contrato donde dos partes intercambian flujos de intereses sobre un mismo nominal: una paga **tasa fija** y la otra paga **tasa variable** (referenciada, en México, a la TIIE). Este cuaderno construye una pequeña cartera de swaps, la valúa en una fecha dada, y además calcula su **Valor en Riesgo (VaR)** con simulación histórica.

### 🗺️ Ruta del cuaderno
1. **Marco teórico** — qué es un swap, la pata fija vs. la pata variable, y cómo se proyecta la tasa de un cupón que todavía no se paga (tasa forward).
2. **Parámetros** — cambia la fecha de valuación, el método de interpolación, el nivel de confianza del VaR y las características de cada swap usando **campos de formulario**.
3. **Carga y alineación de datos** — dos curvas de mercado (descuento y cupón variable) que hay que sincronizar por fecha.
4. **Calendario de flujos y tasas forward** — se construye, cupón por cupón, el flujo de cada swap (esto se deja explícito y sin atajos, para que puedas ver cada pago).
5. **Valuación puntual** — el valor de la cartera hoy.
6. **Valor en Riesgo histórico** — reutilizando las mismas curvas ya cargadas, se revalúa la cartera bajo cada escenario histórico para estimar el VaR y el Expected Shortfall.

> ⚡ **Sobre la eficiencia:** la versión original interpolaba las curvas de mercado para **todas** las fechas históricas solo para usar el resultado de un solo día en la valuación puntual — el resto se calculaba y se tiraba a la basura. Aquí separamos ambos usos: la valuación puntual interpola *solo* el día que necesitas, y el recorrido completo por el historial se hace **una vez**, con un propósito real: la sección de VaR.

**Cómo usarlo:** ejecuta todo con *Entorno de ejecución → Ejecutar todas* la primera vez; después cambia cualquier campo de formulario y vuelve a ejecutar desde ahí hacia abajo.


In [ ]:
# @title 📦 Clonar el repositorio del curso (datos e insumos) { display-mode: "form" }
!rm -rf RiesgosFinancieros
!git clone "https://github.com/pikey-msc/RiesgosFinancieros/"


## 🧮 Marco teórico: ¿cómo se valúa un swap?

En un **IRS** ambas partes se deben flujos sobre el mismo nominal y en las mismas fechas; en la práctica solo se paga la **diferencia neta** en cada fecha de cupón. El valor del swap es:

$$\textrm{IRS}=\textrm{M}\cdot (-1)^z\cdot\sum_{i=1}^n{\frac{(\textrm{t}_{c_{p_{i}}}-\textrm{t}_f)\cdot p_{c_i}/360}{\big(1+\textrm{t}_{vp_{p_{i}}}\cdot p_i/360\big)}}$$

Donde $\textrm{t}_{c_{p_i}}$ es la tasa cupón variable del periodo $i$, $\textrm{t}_f$ la tasa fija pactada, $\textrm{t}_{vp_{p_i}}$ la tasa de descuento al plazo $p_i$, y $z$ indica si se paga fija (0) o variable (1). *(La tabla completa de símbolos está en la sección de Valuación.)*

### El truco: dos curvas, no una

Para valuar un swap necesitamos **dos curvas de mercado**, no una:

- **Curva de descuento** (`tasa_TIIE_SW_OP.txt`) — se usa para traer a valor presente cualquier flujo, sin importar si es fijo o variable.
- **Curva de cupón variable** (`tasa_DIRS_SW_OP.txt`) — se usa *solo* para **proyectar** cuál será la tasa flotante de un cupón que todavía no se paga (no la conocemos con certeza, así que la estimamos con la curva).

### ¿Cómo se proyecta un cupón futuro? La tasa forward

Si conocemos la tasa de descuento a dos plazos, $p_{i-1}$ (el cupón anterior) y $p_i$ (el cupón que queremos proyectar), la tasa forward implícita entre esos dos plazos es:

$$ t_{fwd} = \left[\frac{1+t_{c_{p_i}}\cdot p_i/360}{1+t_{c_{p_{i-1}}}\cdot p_{i-1}/360} - 1\right]\cdot\frac{360}{p_c} $$

Esto es exactamente lo que hace el código más abajo (`XtfwdT` / `interpolar_curvas`), cupón por cupón — **no se colapsa en una fórmula cerrada**, para que quede explícito cómo se proyecta cada pago. El primer cupón de cada swap es la excepción: como ya empezó a devengarse, se usa directo el valor de la curva de cupón variable al plazo del cupón (no hay "cupón anterior" del cual proyectar).

Para interpolar ambas curvas entre nodos usamos la misma función `talamb()` (tasa alambrada) del cuaderno de bonos, o interpolación lineal simple — elegible como campo de formulario.


# 🔧 Funciones auxiliares

In [ ]:
# @title Función talamb(): interpolación por "tasa alambrada" { display-mode: "both" }
import numpy as np

def talamb(nodos, curva, plazos):
    """
    Interpola tasas de interés de forma consistente con las tasas forward
    implícitas entre nodos (método "tasa alambrada", estándar del mercado
    mexicano para curvas gubernamentales).

    Parámetros
    ----------
    nodos  : plazos (en días) donde conocemos la tasa de la curva.
    curva  : tasas de interés observadas en cada nodo.
    plazos : plazos (en días) donde queremos conocer la tasa interpolada.

    Regresa
    -------
    Un arreglo con la tasa interpolada para cada elemento de `plazos`.
    Fuera del rango de `nodos` se sostiene el valor del nodo más cercano
    (no se extrapola).
    """
    nodos = nodos.flatten()
    curva = curva.flatten()
    plazos = plazos.flatten()

    sorted_indices = np.argsort(nodos)
    nodos_sorted = nodos[sorted_indices]
    curva_sorted = curva[sorted_indices]

    indices = np.searchsorted(nodos_sorted, plazos) - 1
    indices[indices < 0] = 0
    indices[indices >= len(nodos_sorted) - 1] = len(nodos_sorted) - 2

    nodos_i = nodos_sorted[indices]
    nodos_i_plus_1 = nodos_sorted[indices + 1]
    TC_interp = curva_sorted[indices]
    TL_interp = curva_sorted[indices + 1]

    TF = np.where(plazos < nodos_sorted[0], curva_sorted[0],
             np.where(plazos > nodos_sorted[-1], curva_sorted[-1],
                      ((((1 + TL_interp * nodos_i_plus_1 / 360) / (1 + TC_interp * nodos_i / 360)) **
                        ((plazos - nodos_i) / (nodos_i_plus_1 - nodos_i)) *
                        (1 + TC_interp * nodos_i / 360)) - 1) * 360 / plazos))

    return TF


# 🎛️ Parámetros de valuación

Aquí defines la fecha de valuación, el método de interpolación, el nivel de confianza para el Valor en Riesgo, y la cartera de swaps a valuar (tasa fija pactada, plazos, periodicidad del cupón, número de contratos, nominal, y si cada swap paga fija o variable).

Como en el cuaderno de bonos, cada bloque es una **celda de formulario** 📋 — cambia los valores en el panel de la derecha y vuelve a correr el cuaderno desde ahí hacia abajo (menú ⋮ de la celda → *Mostrar código* para ver/editar el Python).


In [ ]:
# @title ⚙️ Parámetros generales { display-mode: "form" }
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

fecha_valuacion = '2023-03-10'  # @param {type:"date"}
fval = datetime.strptime(fecha_valuacion.replace('-', ''), "%Y%m%d")  # Fecha de valuación

metodo_interpolacion = "Interpolaci\u00F3n lineal"  # @param ["Interpolaci\u00F3n lineal", "Tasa alambrada"]
itpl = 0 if metodo_interpolacion == "Interpolaci\u00F3n lineal" else 1

nivel_confianza_var = 0.98  # @param {type:"number"}
alpha = nivel_confianza_var  # Nivel de confianza para el Valor en Riesgo (VaR) hist\u00f3rico

print(f"Fecha de valuaci\u00f3n elegida: {fval:%d/%m/%Y}")
print(f"M\u00e9todo de interpolaci\u00f3n: {metodo_interpolacion} (itpl = {itpl})")
print(f"Nivel de confianza para VaR: {alpha:.0%}")


In [ ]:
# @title 🧾 Parámetros — cartera de swaps { display-mode: "form" }
# Curva de descuento (TIIE) y curva de cup\u00f3n variable (DIRS)
btasadesc_sw = "RiesgosFinancieros/2024-1/Tarea/tasa_TIIE_SW_OP.txt"
btasacupvar_sw = "RiesgosFinancieros/2024-1/Tarea/tasa_DIRS_SW_OP.txt"

tasafija_sw = [0.079, 0.075]  # @param {type:"raw"}
plazos_sw = [588, 360]  # @param {type:"raw"}
plazocupon_sw = [28, 28]  # @param {type:"raw"}
contratos_sw = [-1600, 1000]  # @param {type:"raw"}
nominal_sw = [1, 1]  # @param {type:"raw"}
por_sw = [0, 0]  # @param {type:"raw"}

tasafija_sw = np.array(tasafija_sw)        # Tasa fija pactada de cada swap
plazos_sw = np.array(plazos_sw)            # Plazo (d\u00edas) de vida del swap
plazocupon_sw = np.array(plazocupon_sw)    # Periodicidad del cup\u00f3n (d\u00edas)
contratos_sw = np.array(contratos_sw)      # N\u00famero de contratos (posici\u00f3n)
nominal_sw = np.array(nominal_sw)          # Nominal de cada swap
por_sw = np.array(por_sw)                  # 0 = paga fija / recibe variable; 1 = paga variable / recibe fija


# 📥 Carga de datos

Leemos las dos curvas de mercado que necesitamos:

- `tasa_TIIE_SW_OP.txt` → curva de descuento.
- `tasa_DIRS_SW_OP.txt` → curva de cupón variable (para proyectar los cupones futuros).

Cada archivo trae una fila por fecha y una columna por plazo (nodo). Como vienen de fuentes distintas, no necesariamente comparten exactamente las mismas fechas — eso se resuelve en la siguiente sección.


In [ ]:
# @title 📥 Lectura de las curvas de mercado { display-mode: "form" }
data1 = pd.read_table(btasadesc_sw, header=None)
n1 = data1.shape[0]
m1_orig_sw = data1.shape[1]
X1_orig_sw = pd.DataFrame(data1.iloc[1:n1, 1:m1_orig_sw], dtype=float)
X1_orig_sw['Date'] = pd.to_datetime(data1.iloc[1:n1, 0], format='%Y%m%d')
nodos1_sw = data1.iloc[0, 1:m1_orig_sw]

data2 = pd.read_table(btasacupvar_sw, header=None)
n2 = data2.shape[0]
m2_orig_sw = data2.shape[1]
X2_orig_sw = pd.DataFrame(data2.iloc[1:n2, 1:m2_orig_sw], dtype=float)
X2_orig_sw['Date'] = pd.to_datetime(data2.iloc[1:n2, 0], format='%Y%m%d')  # antes usaba n1 por error de dedo
nodos2_sw = data2.iloc[0, 1:m2_orig_sw]

print(f"Curva de descuento:        {n1 - 1} fechas, {len(nodos1_sw)} nodos.")
print(f"Curva de cup\u00f3n variable: {n2 - 1} fechas, {len(nodos2_sw)} nodos.")


## 📆 Alinear las curvas por fecha

Antes de usar ambas curvas juntas, nos quedamos solo con las **fechas que existen en ambos archivos**, y las ordenamos de la más reciente a la más antigua. Así, la fila 0 de cada curva corresponde siempre a la misma fecha en ambas.

*(La versión original resolvía esto con una función genérica para unir una lista de N dataframes por fecha — útil si hubiera muchas curvas, pero aquí solo hay dos, así que lo simplificamos a un `merge` directo.)*


In [ ]:
# @title 📆 Alineación e intersección de fechas { display-mode: "form" }
fechas_comunes = pd.merge(X1_orig_sw[['Date']], X2_orig_sw[['Date']], on='Date', how='inner').drop_duplicates()
fechas_comunes = fechas_comunes.sort_values(by='Date', ascending=False)
n = len(fechas_comunes)
print(f"Fechas en com\u00fan entre ambas curvas: {n}")

def alinear_curva(curva_df, fechas, divide_by_100=True):
    """Se queda solo con las fechas en com\u00fan, ordena de la m\u00e1s reciente a
    la m\u00e1s antigua, y quita la columna de fecha (ya no hace falta)."""
    resultado = pd.merge(fechas, curva_df, on='Date', how='inner').sort_values('Date', ascending=False)
    resultado = resultado.drop(columns=['Date'])
    return resultado / 100 if divide_by_100 else resultado

X1_orig_sw = alinear_curva(X1_orig_sw, fechas_comunes)
X2_orig_sw = alinear_curva(X2_orig_sw, fechas_comunes)


# 🗓️ Calendario de flujos y tasas forward

Para valuar la cartera necesitamos, para cada swap y cada uno de sus cupones:

1. El **plazo acumulado** (días desde hoy) al que cae ese cupón.
2. El plazo acumulado al **cupón anterior** (necesario para proyectar la tasa forward del cupón actual).
3. La **tasa cupón variable proyectada** de ese cupón — vía tasa forward, excepto el primer cupón de cada swap, que ya se conoce directo de la curva.
4. La **tasa de descuento** a la que se trae ese flujo a valor presente.

Esta construcción se deja **cupón por cupón, sin atajos ni fórmulas cerradas** — así puedes inspeccionar el flujo exacto de cualquier swap de la cartera. El código es denso en índices, así que está colapsado como formulario; lo importante es lo que hace cada vector de salida (ver los prints al final de la celda) y la función `interpolar_curvas()`, que reutilizaremos para la valuación puntual y para el VaR histórico sin duplicar código.


In [ ]:
# @title 🗓️ Construcción del calendario de flujos y tasas forward { display-mode: "form" }
def approx(x, y, new_x):
    """Interpolaci\u00f3n lineal simple (equivalente al `approx()` de R)."""
    return np.interp(new_x, x, y.astype(float))

nodosvp = nodos1_sw   # nodos de la curva de descuento
nodostc = nodos2_sw   # nodos de la curva de cup\u00f3n variable
curvavp = X1_orig_sw  # curva de descuento, alineada y ordenada
curvatc = X2_orig_sw  # curva de cup\u00f3n variable, alineada y ordenada

m = len(plazos_sw)
N = (plazos_sw // plazocupon_sw) + 1  # n\u00famero de cupones a pagar por cada swap

# Vectores "aplanados": un elemento por cada flujo (cup\u00f3n) de cada swap
VTplazos_sw = np.zeros(np.sum(N))     # plazo acumulado del flujo
VTplazos_swc = np.zeros(np.sum(N))    # plazo acumulado del cup\u00f3n ANTERIOR (para la tasa forward)
contratos_swT = np.zeros(np.sum(N))
por_swT = np.zeros(np.sum(N))
plazocupon_swT = np.zeros(np.sum(N))
tasafija_swT = np.zeros(np.sum(N))
nominal_swT = np.zeros(np.sum(N))

plazini_sw = plazos_sw - plazocupon_sw * (N - 1)  # plazo del primer cup\u00f3n (puede ser corto)

for j in range(m):
    sum_N = np.sum(N[:j + 1])
    sum_N_prev = np.sum(N[:j]) if j > 0 else 0

    if j == 0:
        VTplazos_sw[:sum_N] = np.arange(plazini_sw[j], plazos_sw[j] + 1, plazocupon_sw[j])
        VTplazos_swc[:sum_N] = np.concatenate(([0], VTplazos_sw[0:(sum_N - 1)]))
        contratos_swT[:sum_N] = contratos_sw[j]
        plazocupon_swT[:sum_N] = plazocupon_sw[j]
        nominal_swT[:sum_N] = nominal_sw[j]
        por_swT[:sum_N] = por_sw[j]
        tasafija_swT[:sum_N] = tasafija_sw[j]
    else:
        VTplazos_sw[sum_N_prev:sum_N] = np.arange(plazini_sw[j], plazos_sw[j] + 1, plazocupon_sw[j])
        VTplazos_swc[sum_N_prev:sum_N] = np.concatenate(([0], VTplazos_sw[sum_N_prev:(sum_N - 1)]))
        contratos_swT[sum_N_prev:sum_N] = contratos_sw[j]
        plazocupon_swT[sum_N_prev:sum_N] = plazocupon_sw[j]
        nominal_swT[sum_N_prev:sum_N] = nominal_sw[j]
        por_swT[sum_N_prev:sum_N] = por_sw[j]
        tasafija_swT[sum_N_prev:sum_N] = tasafija_sw[j]

def interpolar_curvas(i):
    """
    Interpola ambas curvas (descuento y cup\u00f3n variable) al plazo de cada
    flujo, para la fecha de \u00edndice `i` (0 = fecha m\u00e1s reciente), y arma la
    tasa forward de cada cup\u00f3n a partir de la raz\u00f3n de factores de descuento
    entre el cup\u00f3n actual y el anterior. El primer cup\u00f3n de cada swap usa
    directo la curva de cup\u00f3n variable (no hay "cup\u00f3n anterior" del cual
    proyectar). Se usa tanto para la valuaci\u00f3n puntual (una sola fecha) como
    para el VaR hist\u00f3rico (una vez por cada fecha del historial).
    """
    if itpl == 0:
        Xvp_i = approx(nodosvp, curvavp.iloc[i, :], VTplazos_sw)
        Xtc_i = approx(nodostc, curvatc.iloc[i, :], VTplazos_sw)
        Xtcc_i = approx(nodostc, curvatc.iloc[i, :], VTplazos_swc)
    else:
        Xvp_i = talamb(np.array(nodosvp, dtype=float), np.array(curvavp.iloc[i, :], dtype=float), VTplazos_sw)
        Xtc_i = talamb(np.array(nodostc, dtype=float), np.array(curvatc.iloc[i, :], dtype=float), VTplazos_sw)
        Xtcc_i = talamb(np.array(nodostc, dtype=float), np.array(curvatc.iloc[i, :], dtype=float), VTplazos_swc)

    Xtfwd_i = ((1 + Xtc_i * VTplazos_sw / 360) / (1 + Xtcc_i * VTplazos_swc / 360) - 1) * 360 / plazocupon_swT

    j = 0
    while j < len(VTplazos_sw):
        if VTplazos_sw[j] <= plazocupon_swT[j]:
            Xtfwd_i[j] = Xtc_i[j]
        else:
            j = np.sum(N[0:(j + 1)]) - 1
        j += 1

    return Xvp_i, Xtfwd_i

print("Plazo acumulado de cada flujo (d\u00edas):        ", VTplazos_sw)
print("Plazo acumulado del cup\u00f3n anterior (d\u00edas):", VTplazos_swc)


# 💰 Valuación

## Fórmula del swap

$$\textrm{IRS}=\textrm{M}\cdot (-1)^z\cdot\sum_{i=1}^n{\frac{(\textrm{t}_{c_{p_{i}}}-\textrm{t}_f)\cdot p_{c_i}/360}{\big(1+\textrm{t}_{vp_{p_{i}}}\cdot p_i/360\big)}}$$

| Símbolo | Significado |
|---|---|
| $\textrm{IRS}$ | Valor del swap de tasa de interés |
| $\textrm{M}$ | Nominal del contrato |
| $z$ | 0 si se paga tasa fija (se recibe variable); 1 si se paga tasa variable (se recibe fija) |
| $\textrm{t}_{c_{p_i}}$ | Tasa cupón variable proyectada al plazo $p_i$ |
| $\textrm{t}_f$ | Tasa fija pactada |
| $p_{c_i}$ | Plazo del $i$-ésimo cupón (en este curso, todos los cupones de un mismo swap comparten periodicidad) |
| $\textrm{t}_{vp_{p_i}}$ | Tasa de descuento (valor presente) al plazo $p_i$ |
| $p_i$ | Plazo acumulado, en días, al $i$-ésimo cupón |
| $n$ | Número de cupones a pagar |

Esta fórmula descuenta, **cupón por cupón**, la diferencia entre lo que se recibe (tasa variable proyectada) y lo que se paga (tasa fija) — o viceversa, según el signo $z$ de cada swap.


In [ ]:
# @title 🧮 Función y valuación puntual del swap { display-mode: "both" }
def swap(por_swT, contratos_swT, nominal_swT, XtfwdT, tasafija_swT, plazocuponT, VTplazos_sw, Xvp, N):
    """
    Valúa uno o varios swaps sumando, cup\u00f3n por cup\u00f3n, la diferencia entre
    la tasa variable proyectada (`XtfwdT`) y la tasa fija pactada, descontada
    con la curva de descuento (`Xvp`). `N` indica cu\u00e1ntos cupones le
    corresponden a cada swap, en orden.
    """
    V0 = np.zeros(len(N))
    V0f = (contratos_swT * (XtfwdT - tasafija_swT) * (plazocuponT / 360)) / ((1 + Xvp * VTplazos_sw / 360) * nominal_swT * ((-1) ** por_swT))

    for j in range(len(N)):
        if j == 0:
            V0[j] = np.sum(V0f[0:N[j]])
        else:
            V0[j] = np.sum(V0f[np.sum(N[0:j]):np.sum(N[0:(j + 1)])])

    return V0

# Interpolamos las curvas SOLO para la fecha de valuaci\u00f3n (\u00edndice 0 = m\u00e1s reciente)
Xvp_hoy, Xtfwd_hoy = interpolar_curvas(0)
V0_sw = swap(por_swT, contratos_swT, nominal_swT, Xtfwd_hoy, tasafija_swT, plazocupon_swT, VTplazos_sw, Xvp_hoy, N)

resumen_sw = pd.DataFrame({
    'Swap': np.arange(1, m + 1),
    'Plazo (d\u00edas)': plazos_sw,
    'Contratos': contratos_sw,
    'Tasa fija': tasafija_sw,
    'Paga': np.where(por_sw == 0, 'Fija (recibe variable)', 'Variable (recibe fija)'),
    'Valuaci\u00f3n': np.round(V0_sw, 4),
})
resumen_sw


# 📉 Valor en Riesgo (VaR) histórico

Ya tenemos las curvas de mercado de los últimos `n` días y el calendario de flujos de la cartera. En vez de tirar esa información — como hacía la versión original, que interpolaba las `n` fechas y solo usaba una — la aprovechamos para estimar el **riesgo** de la cartera con **simulación histórica**:

1. Revaluamos la cartera **con el mismo calendario de cupones de hoy**, pero usando las curvas de mercado que existieron en cada uno de los últimos `n` días (`interpolar_curvas(i)` para cada `i`).
2. Con esos `n` valores de cartera, construimos la distribución de **pérdidas y ganancias (P&L)** que se habrían observado si las condiciones de mercado de cada día histórico ocurrieran hoy.
3. El **VaR** al nivel de confianza `alpha` es la pérdida que, históricamente, no se superó en esa proporción de los escenarios. El **Expected Shortfall (ES)** es el promedio de las pérdidas que sí lo superan (la "cola" más allá del VaR).

> 📌 Esta es una simulación histórica de un solo factor de revaluación (cambia la curva completa por el escenario histórico; los flujos y la fecha de valuación se mantienen fijos) — el enfoque estándar de VaR histórico para un libro de posiciones.


In [ ]:
# @title 📉 Revaluación histórica y cálculo de VaR / ES { display-mode: "both" }
import matplotlib.pyplot as plt

valores_historicos = np.zeros(n)
for i in range(n):
    Xvp_i, Xtfwd_i = interpolar_curvas(i)
    valores_historicos[i] = np.sum(swap(por_swT, contratos_swT, nominal_swT, Xtfwd_i, tasafija_swT, plazocupon_swT, VTplazos_sw, Xvp_i, N))

# P&L: diferencia contra el valor de hoy (escenario 0), tal como lo ver\u00eda
# alguien parado en la fecha de valuaci\u00f3n
pnl_historico = valores_historicos - valores_historicos[0]

VaR = -np.quantile(pnl_historico, 1 - alpha)
ES = -pnl_historico[pnl_historico <= -VaR].mean()

print(f"Valor de la cartera hoy:            {valores_historicos[0]:,.4f}")
print(f"VaR hist\u00f3rico al {alpha:.0%}:            {VaR:,.4f}")
print(f"Expected Shortfall (ES) al {alpha:.0%}: {ES:,.4f}")

plt.figure(figsize=(9, 5))
plt.hist(pnl_historico, bins=30, color='#4C72B0', alpha=0.85)
plt.axvline(-VaR, color='red', linestyle='--', label=f'VaR {alpha:.0%} = {VaR:,.2f}')
plt.axvline(-ES, color='darkred', linestyle=':', label=f'ES {alpha:.0%} = {ES:,.2f}')
plt.xlabel('P\u00e9rdida / Ganancia de la cartera')
plt.ylabel('Frecuencia (d\u00edas)')
plt.title('Distribuci\u00f3n hist\u00f3rica de P&L de la cartera de swaps')
plt.legend()
plt.tight_layout()
plt.show()


# ✅ Glosario y para seguir practicando

### Glosario rápido
- **IRS (Interest Rate Swap):** contrato donde se intercambian flujos de tasa fija por flujos de tasa variable sobre un mismo nominal.
- **Pata fija / pata variable:** cada uno de los dos flujos que se intercambian en el swap.
- **Curva de descuento:** tasas usadas para traer a valor presente cualquier flujo.
- **Curva de cupón variable:** tasas usadas para proyectar (no descontar) los cupones flotantes que aún no se conocen.
- **Tasa forward:** la tasa implícita entre dos plazos de una curva, usada para estimar el valor de un cupón futuro.
- **VaR (Valor en Riesgo):** la pérdida que, con cierto nivel de confianza, no se espera superar en un horizonte dado.
- **Expected Shortfall (ES):** la pérdida promedio en los escenarios que sí superan el VaR (más sensible a colas extremas).
- **Simulación histórica:** método de VaR/ES que usa escenarios de mercado realmente observados en el pasado, en vez de asumir una distribución teórica.

### 🧪 Ejercicios sugeridos
1. Cambia `nivel_confianza_var` a 0.95 y a 0.99: ¿cómo cambian el VaR y el ES? ¿Por qué el ES siempre es mayor o igual al VaR?
2. Cambia el método de interpolación y compara la valuación puntual y el VaR — ¿el riesgo estimado es sensible al método?
3. Agrega un tercer swap a la cartera (arreglos de 3 elementos en la celda de parámetros) y observa cómo cambia el VaR total contra la suma de los VaR individuales (pista: esto ilustra diversificación).
4. Invierte el signo de `por_sw` de un swap (de pagar fija a pagar variable) y explica qué le pasa a su valuación.
